In [2]:
import requests
import json
from datetime import datetime
from zoneinfo import ZoneInfo
from bs4 import BeautifulSoup


import json

with open("nicar-2026-schedule.json", "r") as f:
    data = json.load(f)

conference_timezone = data.get("timezone", "America/New_York")
sessions_clean = []

for s in data["sessions"]:
    if s.get("canceled"):
        continue

    # Parse ISO time
    start_utc = datetime.fromisoformat(s["start_time"].replace("Z", "+00:00"))
    end_utc = datetime.fromisoformat(s["end_time"].replace("Z", "+00:00"))

    # Convert to conference timezone
    tz = ZoneInfo(conference_timezone)
    start_local = start_utc.astimezone(tz)
    end_local = end_utc.astimezone(tz)

    # Strip HTML from description
    description_html = s.get("description", "")
    description_text = BeautifulSoup(description_html, "html.parser").get_text(" ", strip=True)

    session_obj = {
        "id": s["session_id"],
        "title": s["session_title"],
        "description": description_text,
        "type": s.get("session_type"),
        "skill_level": s.get("skill_level"),
        "room": s.get("room"),
        "tracks": s.get("tracks", []),
        "date": start_local.strftime("%Y-%m-%d"),
        "day": start_local.strftime("%A"),
        "start": start_local.strftime("%H:%M"),
        "end": end_local.strftime("%H:%M"),
        "start_timestamp": start_local.isoformat(),
        "end_timestamp": end_local.isoformat()
    }

    sessions_clean.append(session_obj)

# Sort by time
sessions_clean.sort(key=lambda x: (x["date"], x["start"]))

output = {
    "conference": data["name"],
    "location": data.get("conf_city"),
    "start_date": data.get("start_date"),
    "end_date": data.get("end_date"),
    "sessions": sessions_clean
}

with open("nicar-schedule.json", "w") as f:
    json.dump(output, f, indent=2)

print(f"Saved {len(sessions_clean)} sessions to nicar-schedule.json")


Saved 231 sessions to nicar-schedule.json
